In [1]:
import cups
import os
import json

In [2]:
class PrinterManager:
    def __init__(self):
        try:
            self.conn = cups.Connection()
        except Exception as e:
            raise RuntimeError(f"Impossible de se connecter à CUPS : {e}")

    def lister_imprimantes(self):
        """Retourne une liste des noms d'imprimantes disponibles."""
        try:
            printers = self.conn.getPrinters()
            return list(printers.keys())
        except Exception:
            return []

    def obtenir_statut(self, nom_imprimante):
        """
        CORRECTION : Utilise getPrinters() pour récupérer les attributs.
        Retourne un dictionnaire avec l'état précis.
        """
        try:
            # On récupère toutes les imprimantes et on sélectionne la bonne
            printers = self.conn.getPrinters()
            
            if nom_imprimante not in printers:
                return {"en_erreur": True, "message": "Imprimante introuvable", "bloquee": False}

            attrs = printers[nom_imprimante]
            
            # Récupération des attributs spécifiques
            # printer-state-reasons est une liste de chaînes (ex: ['media-empty-warning'])
            reasons = attrs.get('printer-state-reasons', ['none'])
            state = attrs.get('printer-state', 3) # 3=Idle, 4=Processing, 5=Stopped
            
            status = {
                "nom": nom_imprimante,
                "en_erreur": False,
                "message": "Prête",
                "bloquee": False,
                "reasons": reasons
            }

            # Conversion en une seule chaîne pour la recherche facile
            reasons_str = " ".join(reasons).lower()

            # Logique de détection d'erreurs
            if state == 5: # Stopped
                status["bloquee"] = True
                status["message"] = "En pause (File stoppée)."

            if "media-empty" in reasons_str or "media-needed" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Plus de papier."
            elif "marker-supply-empty" in reasons_str or "no-toner" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Plus d'encre."
            elif "offline" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Imprimante déconnectée."
            elif "input-tray-missing" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Bac papier manquant."
            elif "other-error" in reasons_str:
                status["en_erreur"] = True
                status["message"] = "ERREUR : Problème inconnu."
            elif state == 4:
                status["message"] = "Impression en cours..."
            elif "none" not in reasons_str and reasons:
                 # Cas d'autres erreurs génériques
                 status["message"] = f"Info : {reasons_str}"

            return status

        except Exception as e:
            return {"en_erreur": True, "message": f"Erreur système : {e}", "bloquee": False}

    def reset_error(self, nom_imprimante, purger_file=False):
        """
        Tente de débloquer l'imprimante après une erreur (papier, encre).
        
        :param purger_file: Si True, annule toutes les impressions en attente.
                            Si False, tente de reprendre l'impression là où elle s'est arrêtée.
        """
        print(f"🔄 Tentative de reset sur '{nom_imprimante}'...")

        try:
            # 1. Si on veut tout nettoyer (repartir à zéro)
            if purger_file:
                self.conn.cancelAllJobs(nom_imprimante)
                print("   -> File d'attente purgée (jobs supprimés).")

            # 2. FORCE la réactivation de l'imprimante (commande 'cupsenable')
            # C'est l'étape la plus importante : elle enlève le statut "Paused"
            self.conn.enablePrinter(nom_imprimante)
            print("   -> Imprimante réactivée (Statut 'Enabled').")
            
            # 3. Vérification rapide
            etat = self.obtenir_statut(nom_imprimante)
            if not etat['bloquee']:
                print("✅ Succès : L'imprimante est prête.")
                return True
            else:
                print("⚠️ Attention : L'imprimante semble toujours bloquée (vérifiez l'écran LCD).")
                return False

        except cups.IPPError as e:
            # Souvent une erreur de permissions (Forbidden) si l'utilisateur n'est pas dans le groupe 'lpadmin'
            print(f"❌ Erreur CUPS lors du reset : {e}")
            return False

    def relancer_file(self, nom_imprimante):
        """Réactive l'imprimante (nécessaire après une erreur papier sur Selphy)."""
        try:
            self.conn.enablePrinter(nom_imprimante)
            print(f"-> File d'attente de '{nom_imprimante}' réactivée.")
            return True
        except cups.IPPError:
            return False

    def imprimer_image(self, nom_imprimante, chemin_image, sans_bordure=True, noir_blanc=False):
        """
        Imprime l'image avec gestion des options Selphy CP1500.
        """
        if not os.path.exists(chemin_image):
            raise FileNotFoundError(f"Fichier introuvable : {chemin_image}")
        
        # 1. Options de base (qualité photo)
        options_cups = {
            "MediaType": "photographic",  # Force le mode photo
            "cupsPrintQuality": "Normal", # Qualité standard
            "fit-to-page": "True"         # Adapte l'image à la zone d'impression
        }

        # 2. Gestion du Sans Bordure (Fullbleed)
        # Basé sur ton retour : 'Postcard' (avec marges) vs 'Postcard.Fullbleed' (sans marges)
        # PageSize/Media Size: 54x86mm 54x86mm.Fullbleed 89x119mm 89x119mm.Fullbleed *Postcard Postcard.Fullbleed Custom.WIDTHxHEIGHT
        if sans_bordure:
            options_cups["PageSize"] = "Postcard.Fullbleed"
        else:
            options_cups["PageSize"] = "Postcard"

        # 3. Gestion Noir & Blanc
        if noir_blanc:
            options_cups["ColorModel"] = "Gray"
        else:
            options_cups["ColorModel"] = "RGB"

        try:
            print(f"Envoi avec options : {options_cups}")
            job_id = self.conn.printFile(nom_imprimante, chemin_image, "Job Python", options_cups)
            return job_id
        except cups.IPPError as e:
            raise RuntimeError(f"Erreur d'envoi CUPS : {e}")

In [3]:
pm = PrinterManager()

In [4]:
imprimantes = pm.lister_imprimantes()
print(f"Imprimantes dispos : {imprimantes}")

Imprimantes dispos : ['_maison__Canon_G5000_series', 'Canon_SELPHY_CP1500']


In [5]:
mon_imprimante = "Canon_SELPHY_CP1500" # Remplacez par le nom exact trouvé
etat = pm.obtenir_statut(mon_imprimante)
print(json.dumps(etat, indent=2))

{
  "nom": "Canon_SELPHY_CP1500",
  "en_erreur": false,
  "message": "Pr\u00eate",
  "bloquee": false,
  "reasons": [
    "none"
  ]
}


In [6]:
if mon_imprimante in imprimantes:
    # 2. Check avant impression
    etat = pm.obtenir_statut(mon_imprimante)
    print(f"Statut avant : {etat['message']}")

    if etat['bloquee']:
        print("Réactivation de la file...")
        pm.relancer_file(mon_imprimante)

    # if not etat['en_erreur']:
        # 3. Impression
    try:
        jid = pm.imprimer_image(mon_imprimante, "photo.jpg", sans_bordure=True, noir_blanc=False)
        print(f"Impression lancée ! ID: {jid}")
    except Exception as e:
        print(e)
    # else:
    #     print("Corrigez l'erreur avant d'imprimer.")
else:
    print(f"Imprimante '{mon_imprimante}' non trouvée. Vérifiez le nom et la connexion.")

Statut avant : Prête
Envoi avec options : {'MediaType': 'photographic', 'cupsPrintQuality': 'Normal', 'fit-to-page': 'True', 'PageSize': 'Postcard.Fullbleed', 'ColorModel': 'RGB'}
Impression lancée ! ID: 21


In [ ]:
pm.reset_error(mon_imprimante, purger_file=False)

🔄 Tentative de reset sur 'Canon_SELPHY_CP1500'...
   -> File d'attente purgée (jobs supprimés).
   -> Imprimante réactivée (Statut 'Enabled').
✅ Succès : L'imprimante est prête.


True